# acai.webcrawler — Demo Notebook

Interactive test suite for the hexagonal web-scraping module.  
Run each cell top-to-bottom to verify all components.

> **No browser required** — Selenium is fully mocked in every test cell.

**Architecture under test:**

| Layer | Component | Description |
|-------|-----------|-------------|
| Port | `WebScraperPort` | Abstract interface every adapter implements |
| Domain | `WebConfig` | Configuration value object |
| Domain | `ScraperException` | Exception hierarchy |
| Adapter | `SeleniumScraper` | Chrome WebDriver adapter |
| Factory | `create_scraper()` | Composition root |

## 0 — Setup &amp; Imports

In [ ]:
import sys, io, json
from pathlib import Path
from unittest.mock import MagicMock, patch, PropertyMock
from bs4 import BeautifulSoup

# Ensure the acai package is importable
_project_root = Path.cwd()
while _project_root.name != "acai" and _project_root != _project_root.parent:
    _project_root = _project_root.parent
_shared_python = _project_root.parent  # …/shared/python
if str(_shared_python) not in sys.path:
    sys.path.insert(0, str(_shared_python))

from acai.logging import create_logger, LoggerConfig, LogLevel
from acai.webcrawler import (
    create_scraper,
    WebScraperPort,
    WebConfig,
    ScraperException,
    WebOperationError,
    ConfigurationError,
)
from acai.webcrawler.adapters.outbound.selenium_scraper import SeleniumScraper

# ── Test harness ──────────────────────────────────────────────────────
_results: list[tuple[str, bool, str]] = []

def _record(name: str, passed: bool, detail: str = "") -> None:
    _results.append((name, passed, detail))
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}  {name}" + (f"  — {detail}" if detail else ""))

# Quiet logger for adapter construction
_logger = create_logger(LoggerConfig(service_name="webcrawler-demo", log_level=LogLevel.WARNING))

# ── HTML fixtures ─────────────────────────────────────────────────────
SIMPLE_HTML = """
<html><head><title>Test Page</title></head>
<body><main><p>Main content</p></main></body></html>
"""

CONTENT_DIV_HTML = """
<html><body>
<div id="content"><h1>Title</h1><p>Body text</p></div>
</body></html>
"""

BARE_HTML = "<html><body><p>Only a body</p></body></html>"
EMPTY_HTML = "<html><head></head></html>"


def _make_scraper(config=None) -> SeleniumScraper:
    """Build a SeleniumScraper with a mocked Chrome WebDriver."""
    with patch("acai.webcrawler.adapters.outbound.selenium_scraper.webdriver.Chrome"):
        return SeleniumScraper(logger=_logger, config=config or WebConfig())


print("Imports OK ✔")

Imports OK ✔


## 1 — Factory &amp; Port Contract

Verify that `create_scraper()` returns a `SeleniumScraper` that satisfies the `WebScraperPort` interface.

In [ ]:
with patch("acai.webcrawler.adapters.outbound.selenium_scraper.webdriver.Chrome"):
    scraper = create_scraper(_logger)

is_port = isinstance(scraper, WebScraperPort)
is_selenium = isinstance(scraper, SeleniumScraper)

_record("Factory returns WebScraperPort", is_port, f"type={type(scraper).__name__}")
_record("Factory returns SeleniumScraper", is_selenium)

# With custom config
cfg = WebConfig(headless=True, default_timeout=10)
with patch("acai.webcrawler.adapters.outbound.selenium_scraper.webdriver.Chrome"):
    custom = create_scraper(_logger, cfg)

_record("Factory forwards config", custom._config.headless is True and custom._config.default_timeout == 10)

✅ PASS  Factory returns WebScraperPort  — type=SeleniumScraper
✅ PASS  Factory returns SeleniumScraper
✅ PASS  Factory forwards config


## 2 — WebConfig Validation

Verify that `WebConfig` rejects invalid values with `ConfigurationError`.

In [ ]:
# Defaults are sensible
cfg = WebConfig()
_record("Config defaults", cfg.headless is False and cfg.default_timeout == 3 and cfg.retry_attempts == 3)

# Negative timeout
try:
    WebConfig(default_timeout=-1)
    _record("Config rejects negative timeout", False, "no exception raised")
except ConfigurationError:
    _record("Config rejects negative timeout", True)

# Zero timeout
try:
    WebConfig(default_timeout=0)
    _record("Config rejects zero timeout", False, "no exception raised")
except ConfigurationError:
    _record("Config rejects zero timeout", True)

# Negative page_load_delay
try:
    WebConfig(page_load_delay=-0.5)
    _record("Config rejects negative delay", False, "no exception raised")
except ConfigurationError:
    _record("Config rejects negative delay", True)

# Non-existent driver path
try:
    WebConfig(driver_path="/no/such/chromedriver")
    _record("Config rejects bad driver path", False, "no exception raised")
except ConfigurationError:
    _record("Config rejects bad driver path", True)

✅ PASS  Config defaults
✅ PASS  Config rejects negative timeout
✅ PASS  Config rejects zero timeout
✅ PASS  Config rejects negative delay
✅ PASS  Config rejects bad driver path


## 3 — Exception Hierarchy

Verify that `WebOperationError` and `ConfigurationError` are subclasses of `ScraperException`.

In [ ]:
_record(
    "WebOperationError inherits ScraperException",
    issubclass(WebOperationError, ScraperException),
)
_record(
    "ConfigurationError inherits ScraperException",
    issubclass(ConfigurationError, ScraperException),
)
_record(
    "ScraperException inherits Exception",
    issubclass(ScraperException, Exception),
)

# Catch-all pattern works
try:
    raise WebOperationError("boom")
except ScraperException as exc:
    _record("Catch-all ScraperException works", str(exc) == "boom")

✅ PASS  WebOperationError inherits ScraperException
✅ PASS  ConfigurationError inherits ScraperException
✅ PASS  ScraperException inherits Exception
✅ PASS  Catch-all ScraperException works


## 4 — extract_content (main element)

Verify the scraper finds `&lt;main&gt;`, `#content`, falls back to `&lt;body&gt;`, and reports missing content.

In [ ]:
scraper = _make_scraper()

# <main> element
page = BeautifulSoup(SIMPLE_HTML, "html.parser")
result = scraper.extract_content(page)
_record(
    "Extract <main> content",
    result["content"] is not None and result["length"] > 0 and result["error"] is None,
    f"length={result['length']}",
)

# #content div
page2 = BeautifulSoup(CONTENT_DIV_HTML, "html.parser")
result2 = scraper.extract_content(page2)
_record(
    "Extract #content div",
    result2["content"] is not None and "Title" in result2["content"].get_text(),
)

# Falls back to <body>
page3 = BeautifulSoup(BARE_HTML, "html.parser")
result3 = scraper.extract_content(page3)
_record(
    "Fallback to <body>",
    result3["content"] is not None and "Only a body" in result3["content"].get_text(),
)

# Empty page → error
page4 = BeautifulSoup(EMPTY_HTML, "html.parser")
result4 = scraper.extract_content(page4)
_record(
    "Empty page sets error",
    result4["error"] is not None,
    f"error={result4['error']!r}",
)

✅ PASS  Extract <main> content  — length=32
✅ PASS  Extract #content div
✅ PASS  Fallback to <body>
✅ PASS  Empty page sets error  — error='No content found'


## 5 — get_page (success &amp; URL resolution)

Verify the adapter returns `BeautifulSoup`, and relative URLs are prepended with `base_url`.

In [ ]:
cfg = WebConfig(base_url="https://www.fedlex.admin.ch", page_load_delay=0, retry_delay=0)
scraper = _make_scraper(cfg)
scraper._driver.page_source = SIMPLE_HTML
scraper._wait_for_element = MagicMock(return_value=True)

page = scraper.get_page("https://example.com")
_record(
    "get_page returns BeautifulSoup",
    isinstance(page, BeautifulSoup) and page.title.string == "Test Page",
)

# Relative URL
scraper2 = _make_scraper(cfg)
scraper2._driver.page_source = SIMPLE_HTML
scraper2._wait_for_element = MagicMock(return_value=True)
scraper2.get_page("/eli/cc/24/233_245_233/de")

called_url = scraper2._driver.get.call_args[0][0]
_record(
    "Relative URL prepends base_url",
    called_url == "https://www.fedlex.admin.ch/eli/cc/24/233_245_233/de",
    f"url={called_url}",
)

✅ PASS  get_page returns BeautifulSoup
✅ PASS  Relative URL prepends base_url  — url=https://www.fedlex.admin.ch/eli/cc/24/233_245_233/de


## 6 — get_page (retries)

Verify that `get_page` retries on failure and raises `WebOperationError` when all attempts are exhausted.

In [ ]:
# Succeeds on third attempt
cfg = WebConfig(retry_attempts=3, page_load_delay=0, retry_delay=0)
scraper = _make_scraper(cfg)
scraper._driver.get.side_effect = [Exception("net"), Exception("net"), None]
scraper._driver.page_source = SIMPLE_HTML
scraper._wait_for_element = MagicMock(return_value=True)

page = scraper.get_page("https://example.com")
_record(
    "Retries succeed on 3rd attempt",
    isinstance(page, BeautifulSoup) and scraper._driver.get.call_count == 3,
    f"attempts={scraper._driver.get.call_count}",
)

# All retries exhausted
cfg2 = WebConfig(retry_attempts=2, page_load_delay=0, retry_delay=0)
scraper2 = _make_scraper(cfg2)
scraper2._driver.get.side_effect = Exception("persistent error")

try:
    scraper2.get_page("https://example.com")
    _record("Raises after exhausted retries", False, "no exception raised")
except WebOperationError:
    _record("Raises after exhausted retries", True)

2026-03-23 16:24:14,906 - ERROR - Attempt 1/3 failed: net
2026-03-23 16:24:14,907 - ERROR - Attempt 2/3 failed: net
2026-03-23 16:24:14,909 - ERROR - Attempt 1/2 failed: persistent error
2026-03-23 16:24:14,909 - ERROR - Attempt 2/2 failed: persistent error


✅ PASS  Retries succeed on 3rd attempt  — attempts=3
✅ PASS  Raises after exhausted retries


## 7 — Cleanup &amp; Context Manager

Verify that `cleanup()` quits the driver, is idempotent, and the context manager calls it on exit.

In [ ]:
# cleanup quits driver
scraper = _make_scraper()
mock_driver = scraper._driver
scraper.cleanup()
_record("cleanup() quits driver", mock_driver.quit.called and scraper._driver is None)

# Idempotent
scraper.cleanup()  # should not raise
_record("cleanup() is idempotent", True)

# Context manager
with patch("acai.webcrawler.adapters.outbound.selenium_scraper.webdriver.Chrome"):
    with SeleniumScraper(logger=_logger) as ctx_scraper:
        ctx_driver = ctx_scraper._driver
_record("Context manager calls cleanup", ctx_driver.quit.called)

✅ PASS  cleanup() quits driver
✅ PASS  cleanup() is idempotent
✅ PASS  Context manager calls cleanup


## 8 — Driver Initialisation Failure

Verify that `WebOperationError` is raised when the Chrome WebDriver cannot be created.

In [ ]:
try:
    with patch(
        "acai.webcrawler.adapters.outbound.selenium_scraper.webdriver.Chrome",
        side_effect=Exception("no chrome"),
    ):
        create_scraper(_logger)
    _record("Driver init failure raises", False, "no exception raised")
except WebOperationError as exc:
    _record("Driver init failure raises", True, f"msg={exc}")

2026-03-23 16:24:14,921 - ERROR - Unexpected error during driver initialization: no chrome


✅ PASS  Driver init failure raises  — msg=Unexpected error during driver initialization: no chrome


## 9 — Backward-Compatibility Shim

Verify the `website_scraper` module re-exports the expected names.

In [ ]:
from acai.webcrawler.website_scraper import WebConfig as WC, WebScraper, ScraperException as SE

_record(
    "Shim exports WebConfig",
    WC is WebConfig,
)
_record(
    "Shim exports WebScraper (SeleniumScraper)",
    WC is WebConfig and WebScraper is SeleniumScraper,
)
_record(
    "Shim exports ScraperException",
    SE is ScraperException,
)

✅ PASS  Shim exports WebConfig
✅ PASS  Shim exports WebScraper (SeleniumScraper)
✅ PASS  Shim exports ScraperException


## Summary

In [ ]:
passed = sum(1 for _, ok, _ in _results if ok)
failed = sum(1 for _, ok, _ in _results if not ok)
total = len(_results)

print("=" * 60)
print(f"  RESULTS: {passed}/{total} passed, {failed} failed")
print("=" * 60)

if failed:
    print("\nFailed tests:")
    for name, ok, detail in _results:
        if not ok:
            print(f"  ❌ {name}  — {detail}")
else:
    print("\n🎉 All tests passed!")

  RESULTS: 27/27 passed, 0 failed

🎉 All tests passed!
